<a href="https://colab.research.google.com/github/Guylyan/Guylyan/blob/main/Transcribe.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**1 - Montar o Drive e instalar dependências**

In [ ]:
from google.colab import drive
#from google.colab import userdata

#token_chat = userdata.get('token_chat')
drive.mount('/content/drive')

!pip install faster-whisper ffmpeg-python
#!pip install pyannote.audio
# Se der erro de import depois de instalar, reinicie o runtime (Ambiente de execução > Reiniciar sessão).

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


**2 - Configurações**

In [ ]:
import json
import os
import subprocess
import tempfile

import ctranslate2
from faster_whisper import WhisperModel
#from pyannote.audio import Pipeline
from tqdm import tqdm
from google.colab import userdata

# Reaproveita o token do Hugging Face (o mesmo secret 'HF_TOKEN' usado na diarização) para
# autenticar os downloads do modelo Whisper e evitar o aviso de limite de taxa anônimo.
try:
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    print("Aviso: secret 'HF_TOKEN' não encontrado nos Secrets do Colab — downloads do HF Hub continuarão sem autenticação.")

audio_path = "/content/drive/MyDrive/Colab Notebooks/Audios/Reunião Diretoria X TI 21-08-2026.m4a"

# 'tiny', 'base', 'small', 'medium', 'large-v3', 'large-v3-turbo', 'distil-large-v3'
model_name = "medium"

# Códigos de idioma aceitos pelo faster-whisper (ISO 639-1). "auto" ativa detecção automática.
LANGUAGES = ["auto", "pt", "en", "es", "fr", "de", "it", "ja", "zh"]
language = "pt"  # <- escolha aqui

if language not in LANGUAGES:
    raise ValueError(f"Idioma '{language}' inválido. Opções: {', '.join(LANGUAGES)}")
whisper_language = None if language == "auto" else language

# Contexto para melhorar a grafia de nomes próprios e termos específicos.
initial_prompt ="""
  Reunião de trabalho.
  Participantes:
    Guylyan, Kleyton, Marcos Sato, Cris, Nicolas, Anderson e Alisson.
  Empresa:
    Newglass.
  Termos:
    Orion: Software/Plataforma de BI e Dashboards;
    ConstruShow: ERP da empresa;
    HubSpot: CRM da empresa;
    ViaSoft: empresa desenvolvedora do ConstruShow e Orion;
  """

device_type = "cuda" if ctranslate2.get_cuda_device_count() > 0 else "cpu"
compute_type = "float16" if device_type == "cuda" else "int8"

print(f"Dispositivo: {device_type} | compute_type: {compute_type}")
print(f"Idioma configurado: {language} ({whisper_language or 'auto-detecção'})")

Dispositivo: cpu | compute_type: int8
Idioma configurado: pt (pt)


**3 - Função de conversão/limpeza de áudio**

In [ ]:
def detectar_audio(origem: str) -> dict:
    """Usa ffprobe para identificar a(s) trilha(s) de áudio do arquivo, independente do container
    ou da extensão — funciona tanto para áudio puro (mp3, wav, m4a, ogg, flac...) quanto para
    vídeo com trilha de áudio embutida (mp4, mov, mkv, webm), comum em gravações de Teams/Zoom/Meet."""
    comando = [
        "ffprobe", "-v", "error",
        "-select_streams", "a",
        "-show_entries", "stream=index,codec_name,sample_rate,channels",
        "-of", "json",
        origem,
    ]
    resultado = subprocess.run(comando, capture_output=True, text=True)
    if resultado.returncode != 0:
        raise RuntimeError(f"Falha ao inspecionar o arquivo com ffprobe: {resultado.stderr.strip()}")

    streams = json.loads(resultado.stdout).get("streams", [])
    if not streams:
        raise RuntimeError(
            f"Nenhuma trilha de áudio encontrada em '{origem}'. "
            "Verifique se o arquivo não está corrompido e se realmente contém áudio."
        )

    audio_info = streams[0]
    print(
        f"Áudio detectado: codec={audio_info.get('codec_name')}, "
        f"sample_rate={audio_info.get('sample_rate')}Hz, canais={audio_info.get('channels')}"
    )
    if len(streams) > 1:
        print(f"Aviso: {len(streams)} trilhas de áudio encontradas — usando a primeira.")
    return audio_info


def converter_para_wav(origem: str, destino: str) -> None:
    """Converte para WAV 16kHz mono, com filtros de limpeza (ffmpeg via subprocess).

    - highpass: remove ruído de baixa frequência.
    - afftdn: redução adaptativa de ruído de fundo.
    - loudnorm: normaliza o volume entre falantes.
    O '-map 0:a:0' seleciona explicitamente a primeira trilha de áudio, então funciona igual
    para arquivo de áudio puro ou para vídeo com áudio embutido.
    """
    comando = [
        "ffmpeg", "-i", origem,
        "-map", "0:a:0",
        "-af", "highpass=f=100,afftdn=nf=-25,loudnorm=I=-16:TP=-1.5:LRA=11",
        "-acodec", "pcm_s16le", "-ar", "16000", "-ac", "1",
        destino, "-y", "-loglevel", "quiet",
    ]
    resultado = subprocess.run(comando, capture_output=True, text=True)
    if resultado.returncode != 0 or not os.path.exists(destino) or os.path.getsize(destino) == 0:
        raise RuntimeError(f"Falha ao converter áudio com ffmpeg: {resultado.stderr.strip()}")

**4 - Carregar o modelo**

In [ ]:
if not os.path.isfile(audio_path):
    raise FileNotFoundError(f"Arquivo de áudio não encontrado: {audio_path}")

print(f"Carregando modelo Whisper '{model_name}'...")
model = WhisperModel(model_name, device=device_type, compute_type=compute_type)
print("Modelo carregado.")

Carregando modelo Whisper 'medium'...
Modelo carregado.


**5 - Converter o áudio**

In [ ]:
tmp_dir = tempfile.gettempdir()
wav_audio_path = os.path.join(tmp_dir, os.path.splitext(os.path.basename(audio_path))[0] + ".wav")
processing_audio_path = wav_audio_path

print(f"Inspecionando '{audio_path}'...")
detectar_audio(audio_path)  # levanta erro claro se o arquivo não tiver trilha de áudio

print(f"Convertendo '{audio_path}' para '{wav_audio_path}'...")
try:
    converter_para_wav(audio_path, wav_audio_path)
    print("Conversão concluída.")
except RuntimeError as e:
    print(f"Erro na conversão para WAV: {e}")
    print("Tentando processar o arquivo original diretamente (pode falhar).")
    processing_audio_path = audio_path

Inspecionando '/content/drive/MyDrive/Colab Notebooks/Audios/Reunião Diretoria X TI 21-08-2026.m4a'...
Áudio detectado: codec=aac, sample_rate=48000Hz, canais=2
Convertendo '/content/drive/MyDrive/Colab Notebooks/Audios/Reunião Diretoria X TI 21-08-2026.m4a' para '/tmp/Reunião Diretoria X TI 21-08-2026.wav'...
Conversão concluída.


**6 - Transcrever, com progresso real (sem calibração)**

In [ ]:
segments, info = model.transcribe(
    processing_audio_path,
    language=whisper_language,
    initial_prompt=initial_prompt,
    beam_size=5,
    best_of=5,
    temperature=[0.0, 0.2, 0.4, 0.6, 0.8, 1.0],
    compression_ratio_threshold=2.4,
    log_prob_threshold=-1.0,
    no_speech_threshold=0.5,
    condition_on_previous_text=True,
    vad_filter=True,
    vad_parameters=dict(min_silence_duration_ms=500),
    word_timestamps=True,                 # necessário para o hallucination_silence_threshold funcionar
    hallucination_silence_threshold=2.0,   # descarta texto gerado em silêncios > 2s após corte do VAD
)

print(f"Duração do áudio: {info.duration:.2f}s")
if whisper_language is None:
    print(f"Idioma detectado: {info.language} (confiança {info.language_probability:.2f})")


def formatar_timestamp(segundos: float) -> str:
    """Formata segundos como HH:MM:SS."""
    h = int(segundos // 3600)
    m = int((segundos % 3600) // 60)
    s = int(segundos % 60)
    return f"{h:02d}:{m:02d}:{s:02d}"


base_name = os.path.splitext(os.path.basename(audio_path))[0]
output_dir = os.path.dirname(audio_path) or "."
output_txt_path = os.path.join(output_dir, f"{base_name}.txt")

with open(output_txt_path, "w", encoding="utf-8") as f_out, \
     tqdm(total=info.duration, unit="s", desc="Processando áudio") as pbar:
    ultimo_fim = 0.0
    for segment in segments:
        linha = f"[{formatar_timestamp(segment.start)}] {segment.text.strip()}"
        f_out.write(linha + "\n")
        f_out.flush()
        pbar.update(max(0.0, segment.end - ultimo_fim))
        ultimo_fim = segment.end
    if ultimo_fim < info.duration:
        pbar.update(info.duration - ultimo_fim)

print("Transcrição concluída!")
print(f"Transcrição salva em '{output_txt_path}'")

Duração do áudio: 6065.44s


Processando áudio: 100%|██████████| 6065.4375/6065.4375 [3:10:26<00:00,  1.88s/s]

Transcrição concluída!
Transcrição salva em '/content/drive/MyDrive/Colab Notebooks/Audios/Reunião Diretoria X TI 21-08-2026.txt'


**7 - Limpar arquivo temporário**

In [ ]:
if os.path.exists(wav_audio_path):
    try:
        os.remove(wav_audio_path)
        print(f"Arquivo temporário '{wav_audio_path}' removido.")
    except OSError as e:
        print(f"Erro ao remover arquivo temporário: {e}")

Arquivo temporário '/tmp/Reunião Diretoria X TI 21-08-2026.wav' removido.
